# Cross-Rat LFP Channel Count Audit (All 5 Rats)

**Why this notebook exists:** notebook 011 found Superchris has only 21 LFP channels (missing `T11`),
not the 22 assumed everywhere in every prior notebook. The earlier cross-rat structural audit (notebook
02) only checked the 18 BEHAVIORAL (`bvr`) channel names, it never actually checked LFP channel COUNT or
names per rat. This notebook closes that gap, quickly, before any multi-rat pooled model gets built
around a wrong assumption.

**Likely explanation, not necessarily a data error:** Mitt's channels already ran up to `T23` despite
having only 22 channels (gaps in the numbering scheme, confirmed back in notebook 01), suggesting a
shared, fixed tetrode ID scheme across the lab's hardware, where each rat's file only includes whichever
wires yielded usable signal for that specific animal's implant. A missing channel for one rat is
consistent with a wire that didn't work for that particular surgery, common in chronic multi-electrode
recordings, not necessarily a processing mistake. This notebook checks whether that pattern holds
(gaps are rat-specific and plausible) or looks more like an actual data issue.


In [1]:
import sys
sys.path.append('..')

import numpy as np
from pathlib import Path

raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
print(f"Found {len(session_dirs)} sessions")


Found 5 sessions


## Check LFP Channel Count and Names, Per Rat

In [2]:
channel_lists = {}
for session_dir in session_dirs:
    session_name = session_dir.name
    lfp = np.load(session_dir / f'{session_name}_lfp.npz', allow_pickle=True)
    lfp_keys = lfp['keys'].tolist()
    channel_lists[session_name] = lfp_keys
    print(f"{session_name:20s} {len(lfp_keys)} channels")


080718_mitt          22 channels
081106_barat         22 channels
090212_stella        21 channels
090212_superchris    21 channels
090420_buchanan      20 channels


## Full Channel-by-Channel Comparison

Union of every channel name seen across all 5 rats, with a present/absent grid, so any gaps are
immediately visible, not just a total count difference.


In [3]:
all_channel_names = sorted(set(ch for chs in channel_lists.values() for ch in chs),
                            key=lambda x: int(x.split('_')[0][1:]))

print(f"{'Channel':15s}", end='')
for session_name in channel_lists:
    short_name = session_name.split('_')[1]
    print(f"{short_name:>12s}", end='')
print()

missing_summary = {name: [] for name in channel_lists}
for ch in all_channel_names:
    print(f"{ch:15s}", end='')
    for session_name, chs in channel_lists.items():
        present = ch in chs
        print(f"{'yes' if present else 'MISSING':>12s}", end='')
        if not present:
            missing_summary[session_name].append(ch)
    print()

print("\nSummary of missing channels per rat:")
for session_name, missing in missing_summary.items():
    print(f"  {session_name:20s} missing: {missing if missing else 'none'}")


Channel                mitt       barat      stella  superchris    buchanan
T1_LFP_Raw              yes         yes     MISSING         yes         yes
T2_LFP_Raw              yes         yes         yes         yes         yes
T3_LFP_Raw              yes         yes         yes         yes     MISSING
T4_LFP_Raw              yes         yes         yes         yes         yes
T5_LFP_Raw              yes         yes         yes         yes         yes
T6_LFP_Raw              yes         yes         yes         yes         yes
T7_LFP_Raw              yes         yes         yes         yes         yes
T8_LFP_Raw              yes         yes         yes         yes         yes
T9_LFP_Raw              yes         yes         yes         yes         yes
T10_LFP_Raw             yes         yes         yes         yes         yes
T12_LFP_Raw             yes         yes         yes         yes         yes
T13_LFP_Raw             yes         yes         yes         yes         yes
T14_LFP_Raw 

## Interpretation

If only Superchris shows a gap, and that gap is a single channel (not several), that's consistent with
one bad wire for that specific animal, a normal, expected occurrence, not a pipeline bug. If multiple
rats show different gaps, that further supports "each rat's file only includes usable wires for that
animal" as the general explanation, still not an error, but something the code must handle explicitly
(reading each rat's actual channel count rather than assuming a fixed number).

**Practical fix needed regardless of the cause:** any code that assumes exactly 22 channels (all
notebooks up through 011) needs to read the channel count from each rat's own data instead. This mainly
matters once building a model that pools or compares across rats, a single-rat notebook naturally
already uses that rat's real channel count.


## Text-Only Results Export


In [4]:
import json as _json
import os

results_summary = {
    "purpose": "Cross-rat LFP channel count and identity audit",
    "channel_counts": {name: len(chs) for name, chs in channel_lists.items()},
    "missing_channels_per_rat": missing_summary,
    "all_channel_names_seen_across_any_rat": all_channel_names,
}

print(_json.dumps(results_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook012_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/notebook012_results.json")


{
  "purpose": "Cross-rat LFP channel count and identity audit",
  "channel_counts": {
    "080718_mitt": 22,
    "081106_barat": 22,
    "090212_stella": 21,
    "090212_superchris": 21,
    "090420_buchanan": 20
  },
  "missing_channels_per_rat": {
    "080718_mitt": [],
    "081106_barat": [],
    "090212_stella": [
      "T1_LFP_Raw"
    ],
    "090212_superchris": [
      "T17_LFP_Raw"
    ],
    "090420_buchanan": [
      "T3_LFP_Raw",
      "T14_LFP_Raw"
    ]
  },
  "all_channel_names_seen_across_any_rat": [
    "T1_LFP_Raw",
    "T2_LFP_Raw",
    "T3_LFP_Raw",
    "T4_LFP_Raw",
    "T5_LFP_Raw",
    "T6_LFP_Raw",
    "T7_LFP_Raw",
    "T8_LFP_Raw",
    "T9_LFP_Raw",
    "T10_LFP_Raw",
    "T12_LFP_Raw",
    "T13_LFP_Raw",
    "T14_LFP_Raw",
    "T15_LFP_Raw",
    "T16_LFP_Raw",
    "T17_LFP_Raw",
    "T18_LFP_Raw",
    "T19_LFP_Raw",
    "T20_LFP_Raw",
    "T21_LFP_Raw",
    "T22_LFP_Raw",
    "T23_LFP_Raw"
  ]
}

Saved to outputs/logs/notebook012_results.json
